### DFA-α1 (Jupyter notebook)

End-to-end pipeline:

  1. Load raw RR CSV.
  2. Artifact detection and correction (check artifact_correction.py file for more details).
     -> if percent_artifact > 3%, flag file for exclusion (per Rogers 2021).
  3. Smoothness-priors detrend (lambda = 500).
  4. Time-varying DFA-alpha1: 2-min rolling window, 5s grid step.
  5. Save results to CSV and plot alpha1 + HR over time.

This notebook **imports and calls** the functions in the `.py` files
sitting in this same folder (`data_io.py`, `artifact_correction.py`,
`detrending.py`, `dfa.py`, `windowing.py`).

**Before running:** make sure this notebook is saved in the same folder as
those `.py` files, and that you've run `pip install -r requirements.txt`
once (Cell 2 will also do this for you if you haven't).

In [ ]:
# One-time setup: install required packages
import sys
!{sys.executable} -m pip install -q -r requirements.txt

In [ ]:
# Imports - pulls in the tested pipeline functions from the .py files in this folder
import numpy as np
import matplotlib.pyplot as plt

from data_io import load_rr_csv, remove_offline_dropouts
from artifact_correction import detect_and_correct_artifacts
from detrending import smoothness_priors_detrend
from windowing import time_varying_dfa

# Pipeline settings (matched to Kubios settings commonly used in literature)
ARTIFACT_EXCLUSION_THRESHOLD_PCT = 3.0
DETREND_LAMBDA = 500.0
DFA_LOWER_SCALE = 4
DFA_UPPER_SCALE = 16
WINDOW_SEC = 120.0
STEP_SEC = 5.0

### Step 1 — Load the raw RR data

In [ ]:
CSV_PATH = "hrv.CSV"
df = load_rr_csv(CSV_PATH)
print(f"Loaded {len(df)} beats ({df['rr_ms'].sum()/1000/60:.1f} minutes of recording).")
df.head()

## Step 2 — Remove sensor-dropout rows

Rows where the flag column is True mark a connection dropout (dead time,
not a real heartbeat) — these get removed from the beat sequence entirely.

In [ ]:
df, n_dropped, gap_seconds = remove_offline_dropouts(df)
print(f"Removed {n_dropped} dropout rows ({gap_seconds:.1f}s of dead time). {len(df)} genuine beats remain.")
rr_raw = df["rr_ms"].to_numpy()

## Step 3 — Artifact detection & correction

Approximates Kubios' "automatic" correction method. Prints the percent
of beats flagged; if this exceeds 3%, per Rogers et al. (2021b) this file
should be excluded from further analysis.

In [ ]:
rr_corrected, artifact_mask, pct_artifact = detect_and_correct_artifacts(rr_raw)
print(f"Flagged {artifact_mask.sum()} beats as artifacts ({pct_artifact:.2f}% of total).")

if pct_artifact > ARTIFACT_EXCLUSION_THRESHOLD_PCT:
    print(f"*** EXCLUDE: artifact level {pct_artifact:.2f}% exceeds the "
          f"{ARTIFACT_EXCLUSION_THRESHOLD_PCT}% threshold. Consider stopping here. ***")
else:
    print("Artifact level OK - proceeding.")

## Step 4 — Detrend (smoothness priors, λ = 500)

In [ ]:
rr_detrended = smoothness_priors_detrend(rr_corrected, lam=DETREND_LAMBDA)
print("Detrending done.")

## Step 5 — Time-varying DFA-α1

2-minute rolling window, recalculated every 5 seconds.

In [ ]:
results = time_varying_dfa(
    rr_ms_for_hr=rr_corrected,
    rr_ms_for_dfa=rr_detrended,
    window_sec=WINDOW_SEC,
    step_sec=STEP_SEC,
    lower_scale=DFA_LOWER_SCALE,
    upper_scale=DFA_UPPER_SCALE,
)
results.head()

## Step 6 — Plot

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))

ax1.plot(results["window_center_s"] / 60.0, results["alpha1"], color="tab:blue", label="DFA-alpha1")
ax1.set_xlabel("Time (min)")
ax1.set_ylabel("DFA-alpha1", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.axhline(0.75, color="gray", linestyle="--", linewidth=1, label="alpha1 = 0.75 (aerobic threshold ref.)")

ax2 = ax1.twinx()
ax2.plot(results["window_center_s"] / 60.0, results["mean_hr_bpm"], color="tab:red", alpha=0.6, label="HR (bpm)")
ax2.set_ylabel("Heart rate (bpm)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

fig.suptitle("Time-varying DFA-alpha1 and Heart Rate")
fig.tight_layout()
plt.show()

## Step 7 — Save results

In [ ]:
out_csv = CSV_PATH.rsplit(".", 1)[0] + "_dfa_alpha1_results.csv"
results.to_csv(out_csv, index=False)

out_png = CSV_PATH.rsplit(".", 1)[0] + "_dfa_alpha1_plot.png"
fig.savefig(out_png, dpi=150)

print(f"Saved {out_csv}")
print(f"Saved {out_png}")